In [1]:
import json
import re
from pathlib import Path

import pandas as pd


# --- Akar repo otomatis (tanpa path lokal) ---
def cari_akar_repo(mulai=None):
    mulai = Path.cwd() if mulai is None else Path(mulai)
    for p in [mulai, *mulai.parents]:
        if (p / "pipeline.py").exists() and (p / "src").is_dir():
            return p
    raise FileNotFoundError("Akar repo tidak ditemukan. Buka notebook dari dalam folder repo.")


ROOT = cari_akar_repo()
RAW = ROOT / "data" / "raw"
PROFIL = ROOT / "reports" / "profil"
PROFIL.mkdir(parents=True, exist_ok=True)
print("Akar repo:", ROOT.name)

# --- Seri utama var543 sebagai acuan irisan ---
bps = pd.read_csv(RAW / "bps" / "2026-10-17" / "sdg8_tpt.csv")
TAHUN_BPS = set(bps["tahun"].astype(int))
TURVAR_BPS = sorted(bps["turvar"].astype(str).unique())
print(f"var543: tahun {min(TAHUN_BPS)}-{max(TAHUN_BPS)} | periode {sorted(bps['periode'].astype(str).unique())}")
print("var543 satuan:", sorted(bps["satuan"].astype(str).unique()), "| rincian (turvar):", TURVAR_BPS)

KOLOM = ["sumber", "seri_kode", "nama_seri", "frekuensi", "sex", "age", "dsb", "metode",
         "tahun_min", "tahun_max", "n_obs", "n_catatan", "n_patahan", "n_definisi", "n_sensus",
         "satuan", "irisan_n", "irisan_min", "irisan_max"]


def tambah_irisan(h):
    irisan = h["tahun_set"].apply(lambda s: s & TAHUN_BPS)
    h["irisan_n"] = irisan.apply(len)
    h["irisan_min"] = irisan.apply(lambda s: min(s) if s else pd.NA)
    h["irisan_max"] = irisan.apply(lambda s: max(s) if s else pd.NA)
    return h


# --- ILOSTAT: tiga seri SDMX-CSV ---
SATUAN_ILO = {"PT": "persen"}


def profil_ilo(berkas):
    seri = berkas.stem.replace("DF_", "", 1)
    d = pd.read_csv(berkas, dtype=str)
    d["tahun"] = d["TIME_PERIOD"].str[:4].astype(int)     # '2020' atau '2020-Q3'
    for k in ["SEX", "AGE", "DSB"]:
        if k not in d.columns:
            d[k] = "-"
        d[k] = d[k].fillna("-")
        d["NOTE_INDICATOR"] = d["NOTE_INDICATOR"].fillna("")
    d["SOURCE"] = d["SOURCE"].fillna("")
    d["is_patahan"] = d["NOTE_INDICATOR"].str.contains("Break in series")
    d["is_definisi"] = d["NOTE_INDICATOR"].str.contains("Unemployment definition")
    d["is_sensus"] = d["SOURCE"].str.contains("Census")
    h = d.groupby(["FREQ", "SEX", "AGE", "DSB"]).agg(
        tahun_min=("tahun", "min"), tahun_max=("tahun", "max"),
        tahun_set=("tahun", lambda s: set(s)),
        n_obs=("OBS_VALUE", "count"),
        n_catatan=("NOTE_INDICATOR", lambda s: (s != "").sum()),
        n_patahan=("is_patahan", "sum"), n_definisi=("is_definisi", "sum"),
        n_sensus=("is_sensus", "sum"),
        satuan_kode=("UNIT_MEASURE", "first"),
        metode=("SOURCE", lambda s: " | ".join(sorted(set(x for x in s if x)))),
    ).reset_index()
    h = h.rename(columns={"FREQ": "frekuensi", "SEX": "sex", "AGE": "age", "DSB": "dsb"})
    h["sumber"], h["seri_kode"], h["nama_seri"] = "ilostat", seri, seri
    h["satuan"] = h["satuan_kode"].map(SATUAN_ILO).fillna(h["satuan_kode"])
    return tambah_irisan(h)


ilo = [profil_ilo(b) for b in sorted((RAW / "ilostat" / "2026-10-20" / "data").glob("DF_*.csv"))]

# --- WDI: dua versi angka nasional (bukan duplikat) ---
SERI_WDI = {"SL.UEM.TOTL.ZS": "estimasi model ILO",
            "SL.UEM.TOTL.NE.ZS": "estimasi nasional"}
obs = []
for berkas in sorted((RAW / "wdi" / "2026-10-20" / "data").glob("*.json")):
    with open(berkas, encoding="utf-8") as f:
        isi = json.load(f)
    for o in isi[1]:
        if o["indicator"]["id"] in SERI_WDI and o["value"] is not None:
            obs.append({"seri_kode": o["indicator"]["id"], "nama_seri": o["indicator"]["value"],
                        "tahun": int(o["date"]), "obs_status": o.get("obs_status") or ""})
wdi_raw = pd.DataFrame(obs)
h = wdi_raw.groupby(["seri_kode", "nama_seri"]).agg(
    tahun_min=("tahun", "min"), tahun_max=("tahun", "max"),
    tahun_set=("tahun", lambda s: set(s)), n_obs=("tahun", "count"),
    n_catatan=("obs_status", lambda s: (s != "").sum()),
).reset_index()
h["sumber"], h["frekuensi"], h["sex"], h["age"], h["dsb"] = "wdi", "A", "-", "-", "-"
h["metode"] = h["seri_kode"].map(SERI_WDI)
h["n_patahan"], h["n_definisi"], h["n_sensus"] = pd.NA, pd.NA, pd.NA
h["satuan"] = h["nama_seri"].apply(
    lambda n: (re.findall(r"\(([^)]*)\)", n) or ["tidak tercantum"])[0].replace("% of", "persen dari"))
wdi = tambah_irisan(h)

tabel = pd.concat([*ilo, wdi], ignore_index=True)

# --- Penandaan kandidat sebanding dengan var543 beserta alasan ---
TOTAL_UMUR = ("YGE15", "TOTAL")
UMUR_UTAMA = "AGE_YTHADULT_YGE15"   # klasifikasi umur yang seragam dengan seri SDG dan berentang terpanjang


def nilai_kandidat(r):
    if r["seri_kode"] == "SL.UEM.TOTL.NE.ZS":
        return "ya", ("estimasi nasional (survei) untuk total 15+; metode paling dekat dengan TPT BPS. "
                      "Berbeda dari versi model ILO, bukan duplikat")
    if r["seri_kode"] == "SL.UEM.TOTL.ZS":
        return "ya", ("estimasi model ILO untuk total 15+; pembanding dengan metode berbeda, "
                      "selisih dikategorikan metode atau vintage di tahap 6")
    if r["dsb"] != "-":
        return "tidak", "rincian disabilitas tidak punya padanan di paket BPS; dipakai sebagai konteks"
    if r["frekuensi"] != "A":
        return "tidak", ("triwulanan; var543 berperiode Februari/Agustus/Tahunan, "
                         "perlu aturan pemetaan sebelum dibandingkan")
    if r["sex"] == "SEX_T" and any(t in r["age"] for t in TOTAL_UMUR):
        if r["age"] != UMUR_UTAMA:
            return "tidak", ("total 15+ yang sama dalam klasifikasi umur lain (duplikat cakupan); "
                             f"dipilih {UMUR_UTAMA} karena seragam dengan seri SDG dan berentang terpanjang")
        return "ya", ("total jenis kelamin dan total umur 15+, sepadan dengan cakupan TPT var543; "
                      "bulan survei ILO belum dapat dipastikan dari berkas, diperiksa di tahap 6")
    return "tidak", ("bukan rincian total; var543 tidak punya rincian (hanya total), "
                     "sehingga hanya baris total yang sebanding")


tabel[["kandidat_sebanding", "alasan"]] = tabel.apply(lambda r: pd.Series(nilai_kandidat(r)), axis=1)
def tambah_catatan(r):
    if r["kandidat_sebanding"] != "ya":
        return r["alasan"]
    if r["sumber"] == "ilostat":
        t = (f'{r["alasan"]}; sumber data: {r["metode"]}; dari {r["n_obs"]} observasi, '
             f'{r["n_patahan"]} bertanda patahan seri dan {r["n_definisi"]} bertanda catatan definisi pengangguran')
        if r["n_sensus"] > 0:
            t += f'; {r["n_sensus"]} observasi berasal dari sensus penduduk, bukan survei angkatan kerja'
        return t + "; kecocokan definisi dengan BPS diperiksa di tahap 6"
    return (f'{r["alasan"]}; berkas WDI tidak membawa catatan patahan per observasi, '
            "jadi tidak adanya catatan bukan bukti tanpa patahan")


tabel["alasan"] = tabel.apply(tambah_catatan, axis=1)
tabel = tabel[KOLOM + ["kandidat_sebanding", "alasan"]]

assert tabel["seri_kode"].nunique() == 5, f"Seri ditemukan: {sorted(tabel['seri_kode'].unique())}"
assert (tabel["n_obs"] > 0).all()

tabel.to_csv(PROFIL / "tabel_ketersediaan_seri_pembanding.csv", index=False)
print("Disimpan:", (PROFIL / "tabel_ketersediaan_seri_pembanding.csv").relative_to(ROOT).as_posix())
print(tabel.shape, "| seri:", sorted(tabel["seri_kode"].unique()))
print("\nRingkasan per seri:")
print(tabel.groupby("seri_kode").agg(baris_rincian=("n_obs", "size"), tahun_min=("tahun_min", "min"),
                                      tahun_max=("tahun_max", "max"), satuan=("satuan", "first")).to_string())
print("\nKandidat sebanding:")
print(tabel["kandidat_sebanding"].value_counts().to_string())
print(tabel.loc[tabel["kandidat_sebanding"] == "ya",
                ["seri_kode", "frekuensi", "sex", "age", "tahun_min", "tahun_max", "irisan_n", "satuan"]].to_string())

Akar repo: capstone-dw-kelompok-11
var543: tahun 1986-2026 | periode ['Agustus', 'Februari', 'Tahunan']
var543 satuan: ['Persen'] | rincian (turvar): ['Tidak ada']
Disimpan: reports/profil/tabel_ketersediaan_seri_pembanding.csv
(188, 21) | seri: ['SDG_0852_SEX_AGE_RT', 'SDG_0852_SEX_DSB_RT', 'SL.UEM.TOTL.NE.ZS', 'SL.UEM.TOTL.ZS', 'UNE_DEAP_SEX_AGE_RT']

Ringkasan per seri:
                     baris_rincian  tahun_min  tahun_max                         satuan
seri_kode                                                                              
SDG_0852_SEX_AGE_RT              9       2000       2023                         persen
SDG_0852_SEX_DSB_RT              9       2010       2023                         persen
SL.UEM.TOTL.NE.ZS                1       1976       2023  persen dari total labor force
SL.UEM.TOTL.ZS                   1       1991       2025  persen dari total labor force
UNE_DEAP_SEX_AGE_RT            168       1976       2023                         persen

Kandida

In [7]:
cek = tabel[tabel["kandidat_sebanding"] == "ya"]
print(cek[["seri_kode", "metode", "n_obs", "n_catatan", "n_patahan", "n_definisi", "n_sensus"]].to_string())

print("\nAlasan:")
for _, r in cek.iterrows():
    print("-", r["seri_kode"], ":", r["alasan"])

d = pd.read_csv(RAW / "ilostat" / "2026-10-20" / "data" / "DF_UNE_DEAP_SEX_AGE_RT.csv", dtype=str)
d = d[(d["FREQ"] == "A") & (d["SEX"] == "SEX_T") & (d["AGE"] == "AGE_YTHADULT_YGE15")]
print("\ntahun patahan UNE_DEAP:", sorted(d.loc[d["NOTE_INDICATOR"].fillna("").str.contains("Break in series"), "TIME_PERIOD"]))
print("tahun sensus UNE_DEAP :", sorted(d.loc[d["SOURCE"].fillna("").str.contains("Census"), "TIME_PERIOD"]))

               seri_kode                                                       metode  n_obs  n_catatan n_patahan n_definisi n_sensus
7    SDG_0852_SEX_AGE_RT                           LFS - National Labour Force Survey     24          3         3          0        0
100  UNE_DEAP_SEX_AGE_RT  LFS - National Labour Force Survey | PC - Population Census     42         24         4         23        2
186    SL.UEM.TOTL.NE.ZS                                            estimasi nasional     42          0      <NA>       <NA>     <NA>
187       SL.UEM.TOTL.ZS                                           estimasi model ILO     35          0      <NA>       <NA>     <NA>

Alasan:
- SDG_0852_SEX_AGE_RT : total jenis kelamin dan total umur 15+, sepadan dengan cakupan TPT var543; bulan survei ILO belum dapat dipastikan dari berkas, diperiksa di tahap 6; sumber data: LFS - National Labour Force Survey; dari 24 observasi, 3 bertanda patahan seri dan 0 bertanda catatan definisi pengangguran; kecocokan

In [5]:
d = pd.read_csv(RAW / "ilostat" / "2026-10-20" / "data" / "DF_UNE_DEAP_SEX_AGE_RT.csv", dtype=str)
d = d[(d["FREQ"] == "A") & (d["SEX"] == "SEX_T") & (d["AGE"] == "AGE_YTHADULT_YGE15")]
print("tahun patahan:", sorted(d.loc[d["NOTE_INDICATOR"].fillna("").str.contains("Break in series"), "TIME_PERIOD"]))
print("tahun sensus :", sorted(d.loc[d["SOURCE"].fillna("").str.contains("Census"), "TIME_PERIOD"]))

tahun patahan: ['1990', '2008', '2013', '2021']
tahun sensus : ['1976', '1985']


In [6]:
for nama in ["DF_SDG_0852_SEX_AGE_RT", "DF_UNE_DEAP_SEX_AGE_RT"]:
    d = pd.read_csv(RAW / "ilostat" / "2026-10-20" / "data" / f"{nama}.csv", dtype=str)
    d = d[(d["FREQ"] == "A") & (d["SEX"] == "SEX_T") & (d["AGE"] == "AGE_YTHADULT_YGE15")]
    print("\n==", nama, len(d), "baris")
    print("SOURCE:", d["SOURCE"].value_counts().to_dict())
    print("NOTE_INDICATOR:", d["NOTE_INDICATOR"].value_counts().to_dict())
    print("tahun bercatatan:", sorted(d.loc[d["NOTE_INDICATOR"].notna(), "TIME_PERIOD"]))


== DF_SDG_0852_SEX_AGE_RT 24 baris
SOURCE: {'LFS - National Labour Force Survey': 24}
NOTE_INDICATOR: {'Break in series: Methodology revised': 3}
tahun bercatatan: ['2008', '2013', '2021']

== DF_UNE_DEAP_SEX_AGE_RT 42 baris
SOURCE: {'LFS - National Labour Force Survey': 40, 'PC - Population Census': 2}
NOTE_INDICATOR: {'Unemployment definition: Two criteria (not in employment and seeking)': 20, 'Unemployment definition: Two criteria (not in employment and seeking) | Break in series: Methodology revised': 3, 'Break in series: Methodology revised': 1}
tahun bercatatan: ['1990', '1991', '1992', '1993', '1994', '1996', '1997', '1998', '1999', '2000', '2001', '2002', '2003', '2004', '2005', '2006', '2007', '2008', '2009', '2010', '2011', '2012', '2013', '2021']


In [3]:
cek = tabel[tabel["kandidat_sebanding"] == "ya"]
print(cek[["seri_kode", "metode", "n_obs", "n_catatan", "irisan_n", "irisan_min", "irisan_max"]].to_string())

               seri_kode                                                       metode  n_obs  n_catatan  irisan_n  irisan_min  irisan_max
7    SDG_0852_SEX_AGE_RT                           LFS - National Labour Force Survey     24          3        24        2000        2023
100  UNE_DEAP_SEX_AGE_RT  LFS - National Labour Force Survey | PC - Population Census     42         24        36        1986        2023
186    SL.UEM.TOTL.NE.ZS                                            estimasi nasional     42          0        36        1986        2023
187       SL.UEM.TOTL.ZS                                           estimasi model ILO     35          0        34        1991        2025


In [4]:
import json
import pandas as pd
from pathlib import Path

def cari_akar_repo(mulai=None):
    mulai = Path.cwd() if mulai is None else Path(mulai)
    for p in [mulai, *mulai.parents]:
        if (p / "pipeline.py").exists() and (p / "src").is_dir():
            return p
    raise FileNotFoundError("Akar repo tidak ditemukan.")

ROOT = cari_akar_repo()
RAW = ROOT / "data" / "raw"

for nama in ["DF_SDG_0852_SEX_AGE_RT", "DF_SDG_0852_SEX_DSB_RT", "DF_UNE_DEAP_SEX_AGE_RT"]:
    d = pd.read_csv(RAW / "ilostat" / "2026-10-20" / "data" / f"{nama}.csv")
    print("\n==", nama, d.shape)
    print(d.columns.tolist())
    for c in d.columns:
        if d[c].nunique() <= 30:
            print("  ", c, sorted(d[c].dropna().astype(str).unique()))

w = json.load(open(RAW / "wdi" / "2026-10-20" / "data" / "kelompok-07-hal-01.json", encoding="utf-8"))
print("\n== WDI", type(w), len(w))
print(w[0])
print(w[1][0])


== DF_SDG_0852_SEX_AGE_RT (216, 19)
['DATAFLOW', 'REF_AREA', 'FREQ', 'MEASURE', 'SEX', 'AGE', 'TIME_PERIOD', 'OBS_VALUE', 'OBS_STATUS', 'UNIT_MEASURE_TYPE', 'UNIT_MEASURE', 'UNIT_MULT', 'SOURCE', 'NOTE_SOURCE', 'NOTE_INDICATOR', 'NOTE_CLASSIF', 'DECIMALS', 'UPPER_BOUND', 'LOWER_BOUND']
   DATAFLOW ['ILO:DF_SDG_0852_SEX_AGE_RT(1.0)']
   REF_AREA ['IDN']
   FREQ ['A']
   MEASURE ['SDG_0852_RT']
   SEX ['SEX_F', 'SEX_M', 'SEX_T']
   AGE ['AGE_YTHADULT_Y15-24', 'AGE_YTHADULT_YGE15', 'AGE_YTHADULT_YGE25']
   TIME_PERIOD ['2000', '2001', '2002', '2003', '2004', '2005', '2006', '2007', '2008', '2009', '2010', '2011', '2012', '2013', '2014', '2015', '2016', '2017', '2018', '2019', '2020', '2021', '2022', '2023']
   OBS_STATUS ['B']
   UNIT_MEASURE_TYPE ['RT']
   UNIT_MEASURE ['PT']
   UNIT_MULT ['0']
   SOURCE ['LFS - National Labour Force Survey']
   NOTE_SOURCE ['Repository: ILO-STATISTICS - Micro data processing']
   NOTE_INDICATOR ['Break in series: Methodology revised']
   NOTE_CLASSIF [